# Fazer cruzamento e preencher os buracos

In [ ]:
import pandas as pd
import numpy as np

#  Carregar os dados
df = pd.read_csv('pede_master.csv')

# --- CONTABILIZAÇÃO INICIAL (Para o relatório de resgate) ---
nulos_antes = {
    'data_de_nasc': df['data_de_nasc'].isnull().sum(),
    'escola': df['escola'].isnull().sum(),
    'ipp': df['ipp'].isnull().sum(),
    'indicado_bolsa': df['indicado_bolsa'].isnull().sum(),
    'atingiu_pv': df['atingiu_pv'].isnull().sum()
}

# ---  CRUZAMENTO DE DADOS (RA) ---
cols_mapear = ['data_de_nasc', 'escola', 'ipp', 'indicado_bolsa']
for col in cols_mapear:
    mapa = df.groupby('ra')[col].first()
    df[col] = df[col].fillna(df['ra'].map(mapa))

# --- CÁLCULOS TÉCNICOS ---

# A) IAN (Regra da Defasagem - do PDF)
def calcular_ian(row):
    if 'Fase 8' in str(row['fase']): return 10.0
    d = row['defas']
    if pd.isna(d): return np.nan
    return 10.0 if d >= 0 else (5.0 if d >= -2 else 2.5)

df['ian'] = df.apply(calcular_ian, axis=1)

# B) PONTO DE VIRADA (Baseado no IPV >= 8.0)
df['atingiu_pv'] = df['ipv'].apply(lambda x: 'Sim' if x >= 8.0 else 'Não' if pd.notnull(x) else np.nan)

# C) INDICADO BOLSA (Baseado no corte de INDE >= 6.81)
df.loc[df['indicado_bolsa'].isnull() & df['inde'].notnull(), 'indicado_bolsa'] = \
    df['inde'].apply(lambda x: 'Sim' if x >= 6.81 else 'Não')

# --- RELATÓRIO FINAL COM COMPARAÇÃO ---
print("="*60)
print(f"{'COLUNA':<20} | {'ANTES':<7} | {'DEPOIS':<7} | {'RESGATADOS':<10}")
print("-"*60)

def imprimir_linha(nome_exibicao, col_original):
    antes = nulos_antes[col_original]
    depois = df[col_original].isnull().sum()
    resgatados = antes - depois
    print(f"{nome_exibicao:<20} | {antes:<7} | {depois:<7} | {resgatados:<10}")

imprimir_linha("Data de Nasc", "data_de_nasc")
imprimir_linha("Escola", "escola")
imprimir_linha("IPP", "ipp")
imprimir_linha("Indicado Bolsa", "indicado_bolsa")
imprimir_linha("Ponto de Virada", "atingiu_pv")

print("-"*60)
print(f"IAN: 100% dos {len(df)} registros processados via regra de defasagem.")
print("="*60)

# Salvar
df.to_csv('main.csv', index=False)
print("Arquivo 'main.csv' gerado com sucesso!")

COLUNA               | ANTES   | DEPOIS  | RESGATADOS
------------------------------------------------------------
Data de Nasc         | 860     | 256     | 604       
Escola               | 1875    | 638     | 1237      
IPP                  | 1038    | 431     | 607       
Indicado Bolsa       | 2016    | 29      | 1987      
Ponto de Virada      | 2016    | 178     | 1838      
------------------------------------------------------------
IAN: 100% dos 3030 registros processados via regra de defasagem.
Arquivo 'pede_master_consolidado.csv' gerado com sucesso!
